# Lab 05: Plan & Prepare for MLOps

> **Source:** Microsoft Learning -- [mslearn-mlops](https://github.com/MicrosoftLearning/mslearn-mlops), `docs/05-plan-and-prepare.md`

## Objectives

By the end of this lab you will be able to:

1. **Design** dev/prod workspace separation for ML workloads
2. **Create** a shared Azure ML Registry for cross-workspace model promotion
3. **Configure** data asset isolation between environments
4. **Write** parameterized infrastructure scripts for CI/CD pipelines

| Estimated Time | Estimated Cost |
|---|---|
| ~30 minutes | ~$2-4 |

## Architecture Overview

```mermaid
graph TB
    subgraph "Development"
        DEV_RG[rg-dev] --> DEV_WS[ML Workspace Dev]
        DEV_WS --> DEV_DATA[Sampled Data]
        DEV_WS --> DEV_COMPUTE[Dev Compute]
    end
    subgraph "Production"
        PROD_RG[rg-prod] --> PROD_WS[ML Workspace Prod]
        PROD_WS --> PROD_DATA[Full Data]
        PROD_WS --> PROD_COMPUTE[Prod Compute]
    end
    subgraph "Shared"
        REG[Azure ML Registry]
    end
    DEV_WS -->|register model| REG
    REG -->|promote model| PROD_WS
```

The MLOps environment pattern separates concerns across three layers:

- **Development workspace** -- used for experimentation with sampled data and smaller compute instances. Data scientists iterate here freely.
- **Production workspace** -- runs production workloads against full datasets with autoscaling compute. Tightly controlled access.
- **Shared Azure ML Registry** -- acts as the bridge between environments. Models registered in dev are promoted through the registry to prod, without retraining.

## Why Separate Environments?

In software engineering, separating development and production is standard practice. The same principle applies to ML workloads, but with additional considerations:

| Concern | Development | Production |
|---|---|---|
| **Data** | Sampled/synthetic subset | Full production dataset |
| **Compute** | Small instances, low max nodes | Larger instances, autoscaling |
| **Access** | Data scientists, experimenters | Restricted, CI/CD only |
| **Cost** | Minimal (scale-to-zero) | Optimized for throughput |
| **Governance** | Flexible | Audited, versioned |

The **Azure ML Registry** bridges these environments. It provides a centralized catalog of models, components, and environments that can be shared across workspaces without retraining.

> **Exam Tip:** Azure ML Registry enables cross-workspace model sharing without retraining. A model registered from the dev workspace can be deployed directly in prod via the shared registry. This is a key concept for the DP-100/AI-300 exam.

## Step 1: Provision Dev & Prod Workspaces

The script below creates two Azure resource groups (dev and prod), each with its own Azure ML workspace and compute cluster. The dev cluster has a lower `max-instances` (2) for cost control, while prod allows up to 4 nodes for throughput.

We use a parameterized shell script so that the same logic can be reused across different environments, regions, and naming conventions -- essential for CI/CD pipelines.

In [ ]:
# Display the setup script
with open("scripts/setup_mlops_envs.sh") as f:
    print(f.read())

## Step 2: Create Azure ML Registry

The **Azure ML Registry** is a centralized, cross-workspace catalog for ML assets (models, components, environments). It decouples model development from deployment.

### Create the registry via CLI

```bash
az ml registry create --name <registry-name> --resource-group <rg> --location swedencentral
```

### How promotion works

1. Data scientist trains and registers a model in the **dev** workspace
2. Model is pushed to the **shared registry** (versioned, immutable)
3. CI/CD pipeline pulls the model from the registry into the **prod** workspace
4. Model is deployed to a managed endpoint in prod

This pattern ensures that the exact same model artifact moves from dev to prod -- no retraining, no serialization differences, no "it worked on my machine" surprises.

## Step 3: Data Asset Isolation

Data assets in Azure ML are **workspace-scoped** and **versioned**. This means dev and prod workspaces can reference different datasets independently.

### Dev workspace: sampled data

```bash
az ml data create --name diabetes-dev --type uri_file \
  --path data/diabetes-sample.csv \
  --resource-group rg-ai300-dev \
  --workspace-name mlw-ai300-dev
```

### Prod workspace: full data

```bash
az ml data create --name diabetes-prod --type uri_file \
  --path data/diabetes-full.csv \
  --resource-group rg-ai300-prod \
  --workspace-name mlw-ai300-prod
```

### Why this matters

- **Dev** uses a small sample (e.g., 1,000 rows) for fast iteration
- **Prod** uses the full dataset (e.g., 100,000+ rows) for final training
- Training scripts reference the data asset by name, so the same code runs in both environments -- only the underlying data changes

> **Exam Tip:** Data assets are versioned and workspace-scoped. Creating a new version does not overwrite the previous one, enabling reproducibility and auditing.

## Step 4: Parameterized Scripts for CI/CD

Parameterization is the key to reusable infrastructure. The same script should work for dev, staging, and prod by accepting environment-specific values as arguments or environment variables.

### Pattern: defaults with overrides

```bash
# Set defaults for a specific environment
az configure --defaults group=rg-ai300-dev workspace=mlw-ai300-dev

# Now all subsequent commands use these defaults
az ml job create --file job.yml          # uses rg-ai300-dev / mlw-ai300-dev
az ml model list                         # uses rg-ai300-dev / mlw-ai300-dev
```

### Pattern: environment variables in CI/CD

```yaml
# In GitHub Actions
env:
  RESOURCE_GROUP: rg-ai300-${{ github.event.inputs.environment }}
  WORKSPACE: mlw-ai300-${{ github.event.inputs.environment }}
steps:
  - run: az ml job create --file job.yml -g $RESOURCE_GROUP -w $WORKSPACE
```

### Why this matters

- **One script, many environments** -- no copy-paste, no drift between dev/prod configs
- **Auditable** -- you can trace exactly what parameters were used for each deployment
- **CI/CD-friendly** -- GitHub Actions, Azure DevOps, and other tools can inject values at runtime

## Key Takeaways

1. **Dev/prod separation** -- isolate experimentation from production workloads using separate resource groups and workspaces
2. **Shared Azure ML Registry** -- promotes models between workspaces without retraining; provides a single source of truth for production-ready models
3. **Data asset isolation** -- dev uses sampled data for speed, prod uses full data for accuracy; both are versioned and workspace-scoped
4. **Parameterized scripts** -- write once, deploy everywhere; use arguments or environment variables to target different environments
5. **Environment-specific configuration** -- compute sizes, autoscaling limits, access controls, and data sources all vary by environment

> **Next:** In Lab 06, you will automate this infrastructure provisioning and model training using GitHub Actions CI/CD pipelines.